# R2 archive storage profile
Row counts and Parquet file sizes for every archived UTC day in the Cloudflare R2 archive (`archive/<table>/date=YYYY-MM-DD/data.parquet`). The Raspberry Pi MySQL database keeps only recent days; this archive holds the full history and is written by `python -m data_ingestion.archive`.

Only Parquet footers are read, so the profile downloads a few kilobytes per file. R2 credentials (`R2_*`) are read from the ignored `.env` file at the repository root. Use the `transit_env` kernel.


## Packages

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


## Constants

In [ ]:
repo_root = next(
    (p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (p / 'forecasting' / 'forecasting').is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(repo_root / 'forecasting'))

from forecasting.r2_archive import ARCHIVE_TABLES, archive_uri, connect_r2_archive, list_archive_days

env_file = repo_root / '.env'
table_colors = {'trip_updates': '#2a78d6', 'vehicle_updates': '#eb6834'}


## Fetch archive metadata

In [ ]:
con, bucket = connect_r2_archive(env_file)
frames = []
for table in ARCHIVE_TABLES:
    days = list_archive_days(table, env_file=env_file, con=con)
    files = ', '.join(f"'{archive_uri(bucket, table, day)}'" for day in days)
    metadata = con.execute(
        f'SELECT file_name, num_rows, file_size_bytes FROM parquet_file_metadata([{files}])'
    ).df()
    metadata['day'] = pd.to_datetime(metadata.file_name.str.extract(r'date=(\d{4}-\d{2}-\d{2})')[0])
    frames.append(metadata.assign(table=table))
con.close()

archive_files = (
    pd.concat(frames, ignore_index=True)
    .rename(columns={'num_rows': 'rows', 'file_size_bytes': 'size_bytes'})
    [['table', 'day', 'rows', 'size_bytes']]
    .sort_values(['table', 'day'], ignore_index=True)
)
archive_files['size_mb'] = archive_files.size_bytes / 1e6
print(f'{len(archive_files):,} archived files')


## Summary per table

In [ ]:
summary = archive_files.groupby('table').agg(
    days=('day', 'size'),
    first_day=('day', 'min'),
    last_day=('day', 'max'),
    empty_days=('rows', lambda rows: int((rows == 0).sum())),
    rows=('rows', 'sum'),
    size_mb=('size_mb', 'sum'),
)
summary['bytes_per_row'] = (summary.size_mb * 1e6 / summary.rows).round(1)
summary.loc['total', ['rows', 'size_mb']] = summary[['rows', 'size_mb']].sum()
display(summary.round({'size_mb': 1}))

recent = archive_files[archive_files.day >= archive_files.day.max() - pd.Timedelta(days=27)]
recent_daily_mb = recent.groupby('day').size_mb.sum().mean()
print(f'Recent growth: {recent_daily_mb:.1f} MB/day, about {recent_daily_mb * 30 / 1e3:.2f} GB/month')


## Daily rows and file size
Rows per day use one panel per table because the tables differ about tenfold in volume. Gaps and zero days are ingestion outages, not missing archive files.

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=[f'{table}: rows per UTC day' for table in ARCHIVE_TABLES],
)
for row, table in enumerate(ARCHIVE_TABLES, start=1):
    daily = archive_files[archive_files.table == table]
    fig.add_trace(
        go.Scatter(
            x=daily.day, y=daily.rows, name=table, mode='lines',
            line={'color': table_colors[table], 'width': 2},
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:,} rows<extra>' + table + '</extra>',
        ),
        row=row, col=1,
    )
    fig.update_yaxes(title_text='Rows', row=row, col=1)
fig.update_layout(height=550, showlegend=False, hovermode='x unified', template='plotly_white')
fig.show()

size_fig = go.Figure()
for table in ARCHIVE_TABLES:
    daily = archive_files[archive_files.table == table]
    size_fig.add_trace(
        go.Scatter(
            x=daily.day, y=daily.size_mb, name=table, mode='lines',
            line={'color': table_colors[table], 'width': 2},
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} MB<extra>' + table + '</extra>',
        )
    )
size_fig.update_layout(
    title='Parquet file size per UTC day', yaxis_title='MB', height=350,
    hovermode='x unified', template='plotly_white',
)
size_fig.show()
